# Model info

In [ ]:
import torch
import torch.nn as nn
import time
import platform
import psutil
import GPUtil
from torchsummary import summary
from torch.profiler import profile, ProfilerActivity

from utils.criterion import *


def model_diagnostic_report(model, input_shape=(3, 112, 112), batch_size=32, device=None):
    print("=" * 80)
    print("📊 MODEL DIAGNOSTIC REPORT")
    print("=" * 80)

    # ------------------------------------------------------
    # Device setup
    # ------------------------------------------------------
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"🖥️  Device: {device}")
    print(f"🧠 PyTorch version: {torch.__version__}")
    print(f"💻 System: {platform.system()} {platform.release()}")
    print(f"👤 CPU Cores: {psutil.cpu_count(logical=True)}")
    if device == "cuda":
        gpus = GPUtil.getGPUs()
        for gpu in gpus:
            print(f"⚡ GPU: {gpu.name}, {gpu.memoryTotal} MB VRAM")
    print("-" * 80)

    # ------------------------------------------------------
    # Basic structure
    # ------------------------------------------------------
    print(f"📦 Model structure: {model.__class__.__name__}")
    print(f"🔢 Input shape: {input_shape}")
    print("-" * 80)
    try:
        model.eval()
        summary(model, input_size=input_shape)
    except Exception as e:
        print(f"[!] torchsummary failed: {e}")
    print("-" * 80)

    # ------------------------------------------------------
    # Parameter count
    # ------------------------------------------------------
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"🧮 Total parameters: {total_params:,}")
    print(f"🧩 Trainable parameters: {trainable_params:,}")
    print(f"💾 Estimated model size: {trainable_params * 4 / (1024**2):.2f} MB (float32)")
    print("-" * 80)

    # ------------------------------------------------------
    # FLOPs estimation (torch.profiler)
    # ------------------------------------------------------
    dummy_input = torch.randn(batch_size, *input_shape).to(device)
    dummy_label = torch.randint(0, 10, (batch_size,)).to(device)  # fake label

    model.eval()
    try:
        with profile(
            activities=[ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device == "cuda" else []),
            record_shapes=True,
            with_flops=True
        ) as prof:
            with torch.no_grad():
                # Handle models that require labels even in eval
                try:
                    _ = model(dummy_input, dummy_label)
                except TypeError:
                    _ = model(dummy_input)
        total_flops = sum([evt.flops for evt in prof.key_averages() if hasattr(evt, "flops") and evt.flops is not None])
        print(f"🧠 Estimated FLOPs (torch.profiler): {total_flops / 1e9:.3f} GFLOPs")
    except Exception as e:
        print(f"[!] torch.profiler failed: {e}")
    print("-" * 80)

    # ------------------------------------------------------
    # Timing test
    # ------------------------------------------------------
    model.eval()
    with torch.no_grad():
        start = time.time()
        for _ in range(30):
            try:
                _ = model(dummy_input, dummy_label)
            except TypeError:
                _ = model(dummy_input)
        end = time.time()
    avg_time = (end - start) / 30
    print(f"⏱️  Average forward time (batch{batch_size}): {avg_time * 1000:.2f} ms")
    print("-" * 80)

    # ------------------------------------------------------
    # Memory usage
    # ------------------------------------------------------
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        try:
            _ = model(dummy_input, dummy_label)
        except TypeError:
            _ = model(dummy_input)
        torch.cuda.synchronize()
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        print(f"💾 Peak VRAM usage (forward pass): {peak_mem:.2f} MB")
    else:
        print(f"💾 RAM usage (process): {psutil.Process().memory_info().rss / (1024**2):.2f} MB")
    print("=" * 80)
    print("✅ Profiling complete.")
    print("=" * 80)


# ---------------------- Example usage ----------------------
if __name__ == "__main__":
    # Example model – replace with your own
    model = ArcFaceNet(num_classes=10575)
    model_diagnostic_report(model, input_shape=(3, 112, 112), batch_size=128)


In [6]:
import socket
import subprocess
import requests
import os

from dotenv import load_dotenv
load_dotenv()

def notify_pushover():
    def get_device_name():
        return socket.gethostname()

    def get_gpu_name():
        try:
            return subprocess.check_output(
                "nvidia-smi --query-gpu=name --format=csv,noheader", 
                shell=True, text=True
            ).strip().split('\n')[0]
        except:
            return "No GPU detected"

    
    user_key = os.getenv('PUSHOVER_API_USER_KEY')
    app_token = os.getenv('PUSHOVER_API_TOKEN')
    url = "https://api.pushover.net/1/messages.json"
    message = f"{get_device_name()} - {get_gpu_name()} - finished 🚀"
    data = {"token": app_token, "user": user_key, "message": message}
    requests.post(url, data=data)
    print("### Notification is sent ###")

notify_pushover()

### Notification is sent ###


In [5]:
import socket
import subprocess
import requests
import os

def notify_pushover():
    def get_device_name():
        return socket.gethostname()

    def get_gpu_name():
        try:
            output = subprocess.check_output(
                "nvidia-smi --query-gpu=name --format=csv,noheader",
                shell=True, text=True
            ).strip()
            return output.split('\n')[0] if output else "No GPU detected"
        except subprocess.CalledProcessError:
            return "No GPU detected"
        except FileNotFoundError:
            return "nvidia-smi not found"

    user_key = os.getenv('PUSHOVER_API_USER_KEY')
    app_token = os.getenv('PUSHOVER_API_TOKEN')

    if not user_key or not app_token:
        print("⚠️ Missing Pushover API credentials in environment variables.")
        return

    message = f"{get_device_name()} - {get_gpu_name()} - finished 🚀"
    url = "https://api.pushover.net/1/messages.json"
    data = {"token": app_token, "user": user_key, "message": message}

    try:
        response = requests.post(url, data=data)
        if response.status_code == 200:
            print("✅ Pushover notification sent successfully!")
        else:
            print(f"⚠️ Failed to send notification: {response.status_code} - {response.text}")
    except requests.RequestException as e:
        print(f"❌ Network error while sending Pushover notification: {e}")

if __name__ == "__main__":
    notify_pushover()


⚠️ Missing Pushover API credentials in environment variables.
